# PD02 · Consultar y resumir EVA

[Índice y distribución de las cuatro horas](../ejercicios/README.md) · [Guía de pandas](../docs/manual-pandas.md)

**Tiempo de trabajo: 70 minutos.** El objetivo es traducir una pregunta a filtros verificables y resúmenes. La pregunta de trabajo es: ¿cómo se distribuyen los registros agrícolas de Boyacá entre cultivos durante 2023 y 2024?

| Símbolo | Uso |
|:---:|---|
| 📘 | Concepto o función de apoyo. |
| ✅ | Evidencia que debe quedar visible. |
| ⚠️ | Límite de interpretación. |

Esta plantilla se encuentra en `ejercicios/PD02_consultas_eva.ipynb`. Antes de comenzar, se crea una copia en `soluciones/PD02_consultas_eva.ipynb` dentro del repositorio de cada estudiante. El desarrollo y la entrega se realizan en esa copia; la plantilla de `ejercicios/` se conserva como enunciado. En JupyterLab se puede duplicar el archivo y mover la copia a `soluciones/`, manteniendo el nombre indicado; si ya existe una solución, se continúa sobre ella. Las instrucciones se presentan en celdas Markdown; las celdas de código marcadas como **Desarrollo** contienen el espacio de trabajo. Cada actividad incluye una celda para explicar el resultado.

Se utiliza el kernel del entorno `.venv` del curso. Los CSV ya deben estar descargados mediante `kit/00_datos.py`. La celda inicial prepara las rutas y carga únicamente las entradas de este ejercicio; el desarrollo del análisis corresponde al estudiante. Este notebook se ejecuta de forma independiente.

**Apoyo en el manual:** secciones 4, 5, 8, 10 y las recetas de porcentajes y equivalencia de la sección 11.

## Ubicación de trabajo: la raíz del repositorio

📘 **La raíz del repositorio** es la carpeta principal de la copia del proyecto. En la instalación del curso se llama `big-data-postgraduate` y contiene `README.md`, `kit/`, `ejercicios/` y `soluciones/`:

```text
bigdata/
└── big-data-postgraduate/       ← raíz del repositorio
    ├── README.md
    ├── kit/
    ├── ejercicios/              ← enunciados y plantillas
    └── soluciones/              ← notebooks resueltos
```

**Desde la terminal de Ubuntu en WSL.** `pwd` muestra la carpeta actual, `cd` cambia de carpeta y `ls` muestra su contenido. Los siguientes comandos se ejecutan en la terminal, no en una celda Python:

```bash
pwd
cd /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
pwd
ls
```

✅ Después de `cd`, el segundo `pwd` debe mostrar `/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate`. En la salida de `ls` deben aparecer `README.md`, `kit`, `ejercicios` y `soluciones`. La carpeta `bigdata` es la carpeta contenedora; la raíz del proyecto es su subcarpeta `big-data-postgraduate`.

La ruta anterior corresponde a la configuración de la guía WSL. Si el repositorio se clonó en otra ubicación, se sustituye por la ruta real de esa copia. Si `cd` informa que la carpeta no existe, se revisa esa ubicación antes de continuar.

**Desde una subcarpeta conocida.** `..` significa «carpeta superior». Se comprueba primero la ubicación con `pwd` y se utiliza la alternativa correspondiente:

| Ubicación actual, relativa al repositorio | Comando para volver a la raíz |
|---|---|
| `ejercicios/`, `soluciones/` o `kit/` | `cd ..` |
| `kit/Notebooks/` | `cd ../..` |

Al llegar a la raíz, se puede activar el entorno y abrir JupyterLab desde la terminal:

```bash
source .venv/bin/activate
jupyter lab
```

**Dentro de un notebook ya abierto.** La terminal y el kernel de Python mantienen sus propios directorios de trabajo. La celda de carga siguiente localiza la raíz y la guarda en `REPO`, aunque el notebook esté abierto desde `soluciones/` o `ejercicios/`. Después de ejecutar esa celda, el siguiente bloque puede ejecutarse en una celda Python para comprobar la ubicación y cambiar el directorio del kernel a la raíz:

```python
import os
from pathlib import Path

print("Directorio antes del cambio:", Path.cwd())
print("Raíz detectada:", REPO)
os.chdir(REPO)
print("Directorio después del cambio:", Path.cwd())
```

✅ `Path.cwd()` debe coincidir con `REPO` después del cambio. Esta operación cambia la base para interpretar rutas relativas; el archivo del notebook permanece en `soluciones/`. La carga de datos de estos ejercicios también funciona sin ese cambio porque construye las rutas a partir de `REPO`.

## Preparación

Se ejecuta la siguiente celda y se comprueba que el intérprete pertenece al entorno del curso y que pandas corresponde a la versión 2.2.3. Las rutas se calculan desde el repositorio, sin introducir rutas personales de Windows, WSL o macOS.

In [2]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

actual = Path.cwd().resolve()
REPO = next(
    (ruta for ruta in [actual, *actual.parents]
     if (ruta / "kit/fuentes.json").is_file()
     and (ruta / "kit/00_datos.py").is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError("El notebook debe abrirse dentro del repositorio del curso")

RAW = REPO / "kit/data/raw"
OUT = REPO / "kit/salidas/soluciones/PD02"
archivos = ['eva.csv']
faltantes = [nombre for nombre in archivos if not (RAW / nombre).is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan CSV en {RAW}: {faltantes}")
OUT.mkdir(parents=True, exist_ok=True)

opciones = dict(sep=",", encoding="utf-8-sig", dtype=str, keep_default_na=False)
eva = pd.read_csv(RAW / "eva.csv", **opciones)


print("Python:", sys.version.split()[0])
print("Intérprete:", sys.executable)
print("pandas:", pd.__version__)
print("Repositorio:", REPO)
print("Salidas:", OUT)

Python: 3.12.15
Intérprete: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/.venv/bin/python
pandas: 2.2.3
Repositorio: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
Salidas: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD02


## 1. Preparación de columnas · 10 minutos

Inspeccionar la tabla y crear `codigo_departamento` a partir de `Código Dane departamento`, `anio` desde `Año`, `area_ha` desde `Área cosechada` y `produccion_t` desde `Producción`. El código se conserva como texto; año, área y producción se convierten a tipos numéricos. Registrar las ausencias resultantes.

📘 **Apoyo:** `.str.strip`, `pd.to_numeric`, `astype("Int64")` e `isna`.

✅ **Comprobación:** Las columnas originales se conservan y no se eliminan filas. El corte de entrada contiene 166.732 registros.

In [5]:
# Desarrollo de la actividad 1
# Preparar las cuatro columnas de trabajo en eva.
# Mostrar los tipos y los valores que no pudieron convertirse.

# Crear columnas derivadas
eva["codigo_departamento"] = eva["Código Dane departamento"].str.strip()
eva["anio"] = pd.to_numeric(eva["Año"].str.strip(), errors="coerce").astype("Int64")
eva["area_ha"] = pd.to_numeric(eva["Área cosechada"].str.strip(), errors="coerce")
eva["produccion_t"] = pd.to_numeric(eva["Producción"].str.strip(), errors="coerce")

# Mostrar tipos de datos
print("Tipos de columnas derivadas:\n", eva[["codigo_departamento","anio","area_ha","produccion_t"]].dtypes)

# Registrar ausencias resultantes
ausencias = pd.DataFrame({
    "columna": ["codigo_departamento","anio","area_ha","produccion_t"],
    "nulos": [
        eva["codigo_departamento"].isna().sum(),
        eva["anio"].isna().sum(),
        eva["area_ha"].isna().sum(),
        eva["produccion_t"].isna().sum()
    ]
})

print("\nAusencias detectadas:\n", ausencias)

# Mostrar ejemplos de valores no convertidos
print("\nValores no convertidos en 'anio':")
print(eva.loc[eva["anio"].isna(), "Año"].unique()[:10])

print("\nValores no convertidos en 'area_ha':")
print(eva.loc[eva["area_ha"].isna(), "Área cosechada"].unique()[:10])

print("\nValores no convertidos en 'produccion_t':")
print(eva.loc[eva["produccion_t"].isna(), "Producción"].unique()[:10])

#1. eva["anio"].isna() (Filtro de nulos)
#Crea una máscara booleana (True/False) que identifica qué filas tienen un valor vacío o NaN en la columna "anio".
#2. eva.loc[..., "Año"] (Selección y segmentación)
#El método .loc filtra el DataFrame eva. Selecciona únicamente las filas donde el paso anterior fue True (valores nulos en "anio") y extrae solo el contenido de la columna "Año".

Tipos de columnas derivadas:
 codigo_departamento     object
anio                     Int64
area_ha                float64
produccion_t           float64
dtype: object

Ausencias detectadas:
                columna  nulos
0  codigo_departamento      0
1                 anio      0
2              area_ha      0
3         produccion_t      0

Valores no convertidos en 'anio':
[]

Valores no convertidos en 'area_ha':
[]

Valores no convertidos en 'produccion_t':
[]


**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._

.str.strip() → elimina espacios en blanco antes y después del texto.

pd.to_numeric(..., errors="coerce") → convierte a número; si no puede, marca el valor como NaN.

.astype("Int64") → convierte a entero con soporte para nulos (NaN).

isna().sum() → cuenta los valores faltantes en cada columna derivada.

--------------------------------------------------------
La tabla debe seguir teniendo 166.732 registros.

Las columnas originales (Código Dane departamento, Año, Área cosechada, Producción) se conservan.

Las nuevas columnas (codigo_departamento, anio, area_ha, produccion_t) muestran los tipos correctos: numéricos para año, área y producción. Para el codigo_departamento 
se definio de tipo string pero aparece tipo object.

Los valores que no pudieron convertirse (vacíos, ND, caracteres no numéricos) aparecen como NaN y se registran en la tabla de ausencias.


## 2. Selección con máscara y loc · 15 minutos

Seleccionar el departamento `15` (Boyacá), los años 2023 y 2024, área cosechada positiva y producción no negativa. Guardar todas las columnas de las filas seleccionadas en `consulta_loc`, utilizando `.copy()`. Presentar el total original, el seleccionado y la diferencia.

📘 **Apoyo:** `.eq`, `.isin`, `.gt`, `.ge`, `&` y `.loc`.

✅ **Comprobación:** Todas las filas seleccionadas cumplen simultáneamente las cuatro condiciones. La diferencia respecto a la entrada incluye registros fuera del alcance territorial o temporal; no implica que todos sean errores.

In [6]:
# Desarrollo de la actividad 2
# Construir la máscara y guardar consulta_loc.
# Mostrar los conteos de entrada, selección y diferencia.

# Construir la máscara de condiciones
mask = (
    eva["codigo_departamento"].eq("15") &
    eva["anio"].isin([2023, 2024]) &
    eva["area_ha"].gt(0) &
    eva["produccion_t"].ge(0)
)

# Selección y copia
consulta_loc = eva.loc[mask].copy()

# Conteos
total_original = len(eva)
total_seleccion = len(consulta_loc)
diferencia = total_original - total_seleccion

print("Total original:", total_original)
print("Total seleccionado:", total_seleccion)
print("Diferencia:", diferencia)

Total original: 166732
Total seleccionado: 4924
Diferencia: 161808


**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._

.eq("15") → selecciona solo el departamento con código 15 (Boyacá).

.isin([2023, 2024]) → filtra únicamente los años 2023 y 2024.

.gt(0) → área cosechada estrictamente positiva.

.ge(0) → producción mayor o igual a cero (no negativa).

.loc[mask].copy() → guarda todas las columnas de las filas que cumplen simultáneamente las cuatro condiciones, en un nuevo DataFrame independiente.

----------------------------------------------

El total original es 166.732 registros.

El total seleccionado es menor, solo las filas de Boyacá en 2023–2024 con área > 0 y producción ≥ 0.

La diferencia refleja los registros fuera del alcance territorial o temporal, o con valores que no cumplen las condiciones.

Todas las filas en consulta_loc cumplen simultáneamente las cuatro condiciones, garantizando consistencia.

La diferencia respecto al total original no implica errores, sino que representa registros de otros departamentos, años distintos o valores que no cumplen los filtros. 
La evidencia muestra que la selección es precisa y que la integridad del corte original se mantiene, ya que no se eliminan columnas ni se alteran los datos fuera de la máscara.


## 3. Consulta equivalente con query · 10 minutos

Repetir exactamente la selección anterior mediante `.query`, con `engine="python"`, y guardar el resultado en `consulta_query`. Usar una variable para el código departamental y referenciarla con `@`. Comparar ambos DataFrames sin eliminar columnas.

📘 **Apoyo:** `.query`, `@variable` y `pd.testing.assert_frame_equal`.

✅ **Comprobación:** La comparación debe confirmar las mismas filas, columnas y valores. Si falla, se revisan las condiciones y los tipos antes de continuar.

In [8]:
# Desarrollo de la actividad 3
# Construir consulta_query.
# Comprobar que consulta_loc y consulta_query son equivalentes.

# Variable para el código departamental
codigo = "15"

# Selección con .query y engine="python"
consulta_query = eva.query(
    "codigo_departamento == @codigo and anio in [2023, 2024] and area_ha > 0 and produccion_t >= 0",
    engine="python"
).copy()

# Comparación de ambos DataFrames
import pandas as pd

try:
    pd.testing.assert_frame_equal(consulta_loc, consulta_query)
    print("Los DataFrames son idénticos: mismas filas, columnas y valores.")
except AssertionError as e:
    print("Diferencias detectadas en la comparación:")
    print(e)

Los DataFrames son idénticos: mismas filas, columnas y valores.


**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._

@codigo → referencia la variable externa dentro de la consulta.

.query(..., engine="python") → evalúa la expresión con el motor de Python.

.copy() → asegura que el nuevo DataFrame sea independiente.

pd.testing.assert_frame_equal → confirma que ambos DataFrames tienen las mismas filas, columnas y valores.

-----------------------------------------------------
La comparación valida que la selección con .loc y con .query producen exactamente el mismo subconjunto de datos. Esto garantiza que los filtros están bien definidos y que el procedimiento es reproducible con distintos métodos de pandas. Si hubiera diferencias, sería una señal para revisar la limpieza de datos (espacios, tipos numéricos) antes de avanzar.


## 4. Conteos y porcentajes · 20 minutos

Sobre `consulta_loc`, construir `resumen` con una fila por año y cultivo y una columna `registros`. Añadir el total de registros de cada año y `porcentaje_anual`. Ordenar los resultados por año y cantidad de registros descendente. Conservar los grupos con claves ausentes mediante `dropna=False`.

📘 **Apoyo:** `groupby`, `size`, `reset_index`, `transform("sum")` y `sort_values`.

✅ **Comprobación:** La suma de registros coincide con el tamaño de consulta_loc. Los porcentajes suman aproximadamente 100 % por año; se admite una pequeña tolerancia numérica y se redondea solo para mostrar.

In [14]:
# Desarrollo de la actividad 4
# Crear resumen con registros, total_anio y porcentaje_anual.
# Verificar los totales y los porcentajes por año

# Agrupar por año y cultivo, contar registros
resumen = (
    consulta_loc
    .groupby(["anio", "Cultivo"], dropna=False)
    .size()
    .reset_index(name="registros")
)

# Calcular total por año y porcentaje
resumen["total_anio"] = resumen.groupby("anio")["registros"].transform("sum")
resumen["porcentaje_anual"] = (resumen["registros"] / resumen["total_anio"] * 100).round(2)

# Ordenar resultados
resumen = resumen.sort_values(["anio", "registros"], ascending=[True, False])

print(resumen.head(20))  # muestra las primeras filas

    anio                Cultivo  registros  total_anio  porcentaje_anual
70  2023                   Papa        239        2436              9.81
62  2023                   Maíz        194        2436              7.96
9   2023                 Arveja        176        2436              7.22
38  2023                 Frijol        162        2436              6.65
88  2023                 Tomate        113        2436              4.64
22  2023       Cebolla de bulbo         93        2436              3.82
93  2023                   Yuca         82        2436              3.37
20  2023                   Caña         63        2436              2.59
90  2023                  Trigo         58        2436              2.38
64  2023                   Mora         51        2436              2.09
89  2023        Tomate de árbol         49        2436              2.01
3   2023               Aguacate         48        2436              1.97
8   2023              Arracacha         48        2

**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._

groupby(["anio","Cultivo"], dropna=False) → agrupa por año y cultivo, conservando claves ausentes (NaN).

.size() → cuenta el número de registros en cada grupo.

.reset_index(name="registros") → convierte el resultado en DataFrame con columna registros.

transform("sum") → calcula el total de registros por año y lo asigna a cada fila.

Porcentaje → se obtiene dividiendo los registros de cada grupo sobre el total anual.

sort_values → ordena por año y cantidad de registros descendente.

---------------------------------------------------------
El resumen muestra la distribución de registros por cultivo dentro de cada año.

Permite identificar cuáles cultivos concentran más observaciones en Boyacá durante 2023 y 2024.

El total anual y el porcentaje facilitan comparar la importancia relativa de cada cultivo.

La verificación asegura que no se pierden registros y que los porcentajes son consistentes con el total anual.


## 5. Tabla dinámica y conclusiones · 15 minutos

Construir una tabla dinámica a partir de `resumen`, con cultivos en filas, años en columnas y conteos como valores. Completar con cero las combinaciones sin registros. Exportar `resumen` a `OUT / "resumen_eva.csv"`, sin índice, y redactar dos conclusiones.

📘 **Apoyo:** `pivot_table`, `fill_value=0` y `to_csv`.

✅ **Comprobación:** Los totales de la tabla dinámica concilian con resumen. Las conclusiones se refieren a la cantidad de registros y no a la participación de cada cultivo en la superficie agrícola.

In [15]:
# Desarrollo de la actividad 5
# Construir y mostrar la tabla dinámica.
# Exportar resumen y explicar dos resultados en la celda siguiente.

# Crear tabla dinámica: cultivos en filas, años en columnas, conteos como valores
tabla_dinamica = (
    resumen.pivot_table(
        index="Cultivo",
        columns="anio",
        values="registros",
        fill_value=0,
        aggfunc="sum"
    )
)

# Mostrar la tabla dinámica
print(tabla_dinamica.head(20))

# Exportar resumen a OUT
ruta_resumen = OUT / "resumen_eva.csv"
resumen.to_csv(ruta_resumen, index=False, encoding="utf-8-sig")

print(f"Resumen exportado a: {ruta_resumen}")

anio             2023  2024
Cultivo                    
Acelga              5     8
Achira              4     4
Agraz - mortiño     3     4
Aguacate           48    52
Ahuyama            23    26
Ajo                 7     7
Ají                 1     2
Apio                0     1
Arazá               3     4
Arracacha          48    51
Arveja            176   168
Arándano           18    20
Avena              14    15
Banano             10    11
Berenjena           1     1
Borojó              2     2
Brevo               6     7
Brócoli             5     6
Cacao              21    23
Café               48    50
Resumen exportado a: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD02/resumen_eva.csv


**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._

pivot_table → reorganiza los datos: cultivos como filas, años como columnas, y los conteos (registros) como valores.

fill_value=0 → completa con cero las combinaciones sin registros.

aggfunc="sum" → asegura que los conteos se sumen correctamente si hubiera duplicados.

to_csv(..., index=False) → exporta el DataFrame resumen a CSV sin índice, en la carpeta OUT.

-------------------------------------------------------
Conclusiones

Los registros de Boyacá en 2023 y 2024 se distribuyen en varios cultivos, lo que refleja un foco de seguimiento en esos productos. Se puede analizar cuales son los productos que mas se cosechan y cuales los que menos se cosechan con el fin optar medidas de apoyo a bajas producciones o establecer potenciamiento de mercado en exportación por ejemplo.

La conciliación de totales confirma que no se perdieron registros en la transformación; las combinaciones sin datos se representan explícitamente con cero, lo que facilita análisis comparativos entre años.

## Entrega

El notebook completado se conserva como `soluciones/PD02_consultas_eva.ipynb`. La salida regenerable es `kit/salidas/soluciones/PD02/resumen_eva.csv`.

✅ Deben quedar visibles la equivalencia de consultas, la conciliación de los conteos, los porcentajes por año, la tabla dinámica y dos conclusiones.

⚠️ Una fila EVA es una observación de la fuente, no un predio ni una persona. Un porcentaje de registros no equivale a un porcentaje de área cultivada. El cero de una tabla de conteos indica que no se encontraron filas para esa combinación.